# BENZI Colab

**T4 GPU.** If merge stuck 30+ min → **Step 3B** (download adapter, merge on Mac).


In [ ]:
# GPU
!nvidia-smi
import torch
assert torch.cuda.is_available()


In [ ]:
import os, shutil, subprocess
from pathlib import Path
os.chdir("/content")
REPO = Path("/content/final-year-benzi")
if REPO.exists(): shutil.rmtree(REPO)
subprocess.check_call(["git","clone","--depth","1","https://github.com/sameedsaeed123/final-year-benzi.git",str(REPO)])
os.chdir(REPO / "fyp-ml-demos")
print("cwd", Path.cwd())


In [ ]:
import subprocess, sys
from pathlib import Path
def pip(*a): subprocess.check_call([sys.executable,"-m","pip","install","-q",*a])
pip("-r","requirements.txt")
if Path("requirements-finetune-colab.txt").is_file(): pip("-r","requirements-finetune-colab.txt")
pip("bitsandbytes>=0.43.0","accelerate","peft","datasets")
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"], check=False)


In [ ]:
!python finetune/prepare_dataset.py --max-total 1200


In [ ]:
!python finetune/train_qlora.py --model Qwen/Qwen2.5-3B-Instruct --max-steps 60 --max-length 384 --batch-size 1 --grad-accum 4


In [ ]:
# Step 3A — merge (15–40 min). NO Ctrl+C.
import shutil, subprocess, sys
from pathlib import Path
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"], check=False)
print("Disk GB free:", shutil.disk_usage("/content").free/1e9)
!python finetune/merge_lora_colab.py --model Qwen/Qwen2.5-3B-Instruct


In [ ]:
# Step 3B — stuck? Download adapter ~100MB, merge on Mac
# !python finetune/zip_adapter_for_mac.py
# from google.colab import files
# files.download("/content/benzi-lora-adapter.zip")


In [ ]:
from pathlib import Path
from google.colab import files
import shutil
m = Path("finetune/merged/benzi-empathetic-hf")
assert (m/"config.json").is_file(), "Run 3A first or 3B on Mac"
shutil.make_archive("/content/benzi-empathetic-trained","zip",m)
files.download("/content/benzi-empathetic-trained.zip")


Mac: unzip → `ollama create benzi-empathetic-trained -f Modelfile` → `OLLAMA_MODEL=benzi-empathetic-trained`
